# V7_C_N02 — Teachers Where Learners Need Them

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft using synthetic data. Outputs support authorized human review; they do not constitute official declarations or automated decisions.

## Decision contract
Support transparent teacher-deployment planning by subject, grade, geography, workload, and school capacity. Owner: education ministry and authorized subnational authorities. The model does not transfer staff automatically.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7202); n=48; df=pd.DataFrame({'school':[f'S{i:03d}' for i in range(n)],'district':rng.choice(['D1','D2','D3','D4'],n),'enrolment':rng.integers(120,1100,n),'teachers':rng.integers(5,38,n),'qualified_share':rng.uniform(.55,1,n),'rural':rng.choice([0,1],n,p=[.55,.45]),'classrooms':rng.integers(4,30,n)}); df.head().round(2)

## Evidence contract
Headcounts, full-time equivalents, qualifications, subject assignments, absences, enrolment reference dates, and school status must be reconciled. A pupil-teacher ratio is not a complete measure of teaching need.

In [2]:
assert df.school.is_unique; df['ptr']=df.enrolment/df.teachers; df['qualified_teachers']=df.teachers*df.qualified_share; df['pqr']=df.enrolment/df.qualified_teachers.clip(lower=1); print(df[['ptr','pqr']].describe().round(1))

         ptr    pqr
count   48.0   48.0
mean    46.4   63.0
std     42.3   66.0
min      4.8    6.2
25%     15.3   22.1
50%     35.3   42.4
75%     55.4   71.7
max    174.8  285.6


## Transparent staffing requirement
The planning norm is an explicit policy assumption. It must be differentiated where subject, grade, language, inclusion, or remoteness changes workload.

In [3]:
target_ptr=32; df['required_teachers']=np.ceil(df.enrolment/target_ptr).astype(int); df['gap']=df.required_teachers-df.teachers; print(df.nlargest(8,'gap')[['school','district','enrolment','teachers','required_teachers','gap','rural']].to_string(index=False))

school district  enrolment  teachers  required_teachers  gap  rural
  S008       D2       1024         7                 32   25      0
  S038       D4       1028         8                 33   25      0
  S042       D3        965         7                 31   24      0
  S016       D2        874         5                 28   23      1
  S037       D1        939         7                 30   23      0
  S021       D3        956         8                 30   22      0
  S003       D3       1043        13                 33   20      0
  S007       D2        786         9                 25   16      1


## Equity-aware prioritization
Need combines staffing gap, qualification pressure, and rural/remoteness policy. The weights are reviewable allocation assumptions—not hidden predictions.

In [4]:
df['gap_rate']=(df.gap.clip(lower=0)/df.required_teachers.clip(lower=1)); df['qualification_gap']=1-df.qualified_share; df['priority']=.55*df.gap_rate+.30*df.qualification_gap+.15*df.rural; df.nlargest(8,'priority')[['school','district','priority','gap','qualified_share','rural']].round(2)

## Constrained deployment scenario
Allocate a limited pool one post at a time to the highest current priority, with a cap to avoid unrealistic concentration. This is a scenario for negotiation and review.

In [5]:
pool=24; alloc={s:0 for s in df.school}; work=df.copy();
for _ in range(pool):
 work['remaining_gap']=(work.required_teachers-work.teachers-work.school.map(alloc)).clip(lower=0); eligible=work[work.remaining_gap>0].copy();
 if eligible.empty: break
 eligible['dynamic']=eligible.priority*(eligible.remaining_gap/eligible.required_teachers); chosen=eligible.sort_values(['dynamic','school'],ascending=[False,True]).iloc[0].school; alloc[chosen]+=1
df['allocated']=df.school.map(alloc); df['post_gap']=df.gap-df.allocated; print(df[df.allocated>0].sort_values('allocated',ascending=False)[['school','district','gap','allocated','post_gap']].head(12).to_string(index=False))

school district  gap  allocated  post_gap
  S016       D2   23          9        14
  S008       D2   25          4        21
  S042       D3   24          4        20
  S037       D1   23          3        20
  S007       D2   16          2        14
  S038       D4   25          2        23


## Sensitivity and decision product
Compare allocations with and without the rural factor. Large changes require explicit policy discussion.

In [6]:
alt=.65*df.gap_rate+.35*df.qualification_gap; overlap=len(set(df.nlargest(12,'priority').school)&set(df.assign(alt=alt).nlargest(12,'alt').school))/12; product={'posts_available':pool,'posts_allocated':int(df.allocated.sum()),'priority_overlap':overlap,'status':'SCENARIO—REQUIRES HR AND POLICY REVIEW','prohibited_use':'automatic involuntary transfer'}; print(product)

{'posts_available': 24, 'posts_allocated': 24, 'priority_overlap': 0.9166666666666666, 'status': 'SCENARIO—REQUIRES HR AND POLICY REVIEW', 'prohibited_use': 'automatic involuntary transfer'}


## Exercises
1. Add subject-specific shortages. 2. Use full-time equivalents. 3. Add housing/transport constraints. 4. Explain why PTR improvement is not proof of learning impact.

## Exact solutions
1. Build school-subject-grade requirements and qualification-compatible supply. 2. Replace headcounts with contracted/available teaching time and document absence assumptions. 3. Model feasibility and costs without penalizing underserved schools. 4. Learning depends on pedagogy, attendance, materials, class composition, leadership, and implementation; causal impact needs evaluation.

In [7]:
assert df.allocated.sum()<=pool and product['status'].startswith('SCENARIO'); print('V7_C_N02_COMPLETE_EXECUTION_PASS')

V7_C_N02_COMPLETE_EXECUTION_PASS
